# 高雄各級民代對「橋頭糖廠百頃森林／高雄新市鎮三期」議題立場與發言爬蟲

本 Notebook 專為 Google Colab 設計，協助系統性盤點**現任高雄民意代表**（上至全國立法委員、高雄市議員，下至橋頭及周邊區里長）對於 [守護橋頭糖廠森林](https://maskywg.github.io/qiaotou-forest/) 議題之公開發言與立場。

### 功能架構
1. **步驟 1：安裝與環境設定**
2. **步驟 2：取得現任高雄民代名單**（自動嘗試從立法院／高雄市議會／政府開放資料抓取，並內建完整備援名單避免政府網站阻擋 Colab IP）
3. **步驟 3：Google News RSS + 網頁內文爬蟲**（針對每位民代搜尋「橋頭糖廠」「高雄新市鎮三期」「1-1道路」「森林阿嬤」等關鍵字）
4. **步驟 4：立場初判與（選填）Gemini AI 自動摘要**
5. **步驟 5：匯出 CSV / JSON（可直接匯入網頁前端使用）並繪製統計圖**

In [ ]:
# @title 步驟 1：安裝必要套件
!pip install -q requests beautifulsoup4 feedparser pandas trafilatura tqdm matplotlib

In [ ]:
# @title 步驟 2：匯入套件與設定搜尋關鍵字
import json
import re
import time
from urllib.parse import quote

import feedparser
import pandas as pd
import requests
import trafilatura
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

# 議題核心關鍵字群
ISSUE_KEYWORDS = [
    "橋頭糖廠",
    "高雄新市鎮三期",
    "新市鎮第三期",
    "1-1道路",
    "百頃森林",
    "森林阿嬤",
    "蔣黃女",
]

# 用於 Google News RSS 的複合查詢條件
ISSUE_QUERY = '("橋頭糖廠" OR "高雄新市鎮三期" OR "新市鎮第三期" OR "1-1道路" OR "百頃森林" OR "森林阿嬤")'

### 步驟 3：取得現任高雄民代名單（立委、議員、里長）
這裡會先嘗試連線公開頁面抓取最新名單；若政府網站對 Colab 雲端 IP 有限制或改版，會自動退回使用內建的現任名單並支援自行擴充（例如上傳里長名單 CSV）。

In [ ]:
# @title 3-1 抓取與整理高雄民代名單

def get_kaohsiung_legislators():
    """取得現任高雄相關立委名單（區域 8 席 + 高雄深耕/不分區代表）"""
    # 第 11 屆（2024–2028）高雄市區域立委與高雄相關不分區立委
    legislators = [
        {"層級": "全國立委", "選區": "高雄市第1選區（大旗美等）", "姓名": "邱議瑩", "黨籍": "民主進步黨"},
        {"層級": "全國立委", "選區": "高雄市第2選區（橋頭、岡山、路竹等）", "姓名": "邱志偉", "黨籍": "民主進步黨"},
        {"層級": "全國立委", "選區": "高雄市第3選區（左營、楠梓）", "姓名": "李柏毅", "黨籍": "民主進步黨"},
        {"層級": "全國立委", "選區": "高雄市第4選區（仁武、鳥松、大寮、林園）", "姓名": "林岱樺", "黨籍": "民主進步黨"},
        {"層級": "全國立委", "選區": "高雄市第5選區（三民、苓雅部分）", "姓名": "李昆澤", "黨籍": "民主進步黨"},
        {"層級": "全國立委", "選區": "高雄市第6選區（鼓山、鹽埕、前金、新興、苓雅部分）", "姓名": "黃捷", "黨籍": "民主進步黨"},
        {"層級": "全國立委", "選區": "高雄市第7選區（鳳山）", "姓名": "許智傑", "黨籍": "民主進步黨"},
        {"層級": "全國立委", "選區": "高雄市第8選區（前鎮、小港、旗津）", "姓名": "賴瑞隆", "黨籍": "民主進步黨"},
        {"層級": "全國立委（不分區）", "選區": "高雄責任區／市長候選人", "姓名": "柯志恩", "黨籍": "中國國民黨"},
        {"層級": "全國立委（不分區）", "選區": "高雄相關", "姓名": "陳菁徽", "黨籍": "中國國民黨"},
        {"層級": "全國立委（不分區）", "選區": "高雄相關", "姓名": "林國成", "黨籍": "台灣民眾黨"},
    ]
    return legislators


def crawl_kaohsiung_councilors():
    """嘗試從高雄市議會官網抓取現任議員名單，失敗時使用重點選區備援名單"""
    url = "https://www.kcc.gov.tw/Member/MemberList.aspx"
    councilors = []
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        # 依高雄市議會常見 DOM 結構解析議員姓名
        for a in soup.select("a[href*='MemberDetail'], .member-name, .councilor-name"):
            name = a.get_text(strip=True)
            if 2 <= len(name) <= 5 and not any(k in name for k in ["更多", "詳細", "選區"]):
                councilors.append({
                    "層級": "高雄市議員",
                    "選區": "高雄市議會（官網抓取）",
                    "姓名": name,
                    "黨籍": "待補",
                })
        # 去除重複姓名
        seen = set()
        unique = []
        for c in councilors:
            if c["姓名"] not in seen:
                seen.add(c["姓名"])
                unique.append(c)
        councilors = unique
    except Exception as e:
        print(f"[提示] 高雄市議會官網抓取未取得完整資料（{e}），啟用備援名單。")

    if len(councilors) < 5:
        # 第3選區（橋頭、岡山、燕巢、永安、彌陀、梓官）與第4選區（左營、楠梓）及曾公開表態議員
        councilors = [
            {"層級": "高雄市議員", "選區": "第3選區（岡山、橋頭等）", "姓名": "黃秋媖", "黨籍": "民主進步黨"},
            {"層級": "高雄市議員", "選區": "第3選區（岡山、橋頭等）", "姓名": "林志誠", "黨籍": "民主進步黨"},
            {"層級": "高雄市議員", "選區": "第3選區（岡山、橋頭等）", "姓名": "方信淵", "黨籍": "中國國民黨"},
            {"層級": "高雄市議員", "選區": "第3選區（岡山、橋頭等）", "姓名": "宋立彬", "黨籍": "中國國民黨"},
            {"層級": "高雄市議員", "選區": "第3選區（岡山、橋頭等）", "姓名": "陸淑美", "黨籍": "中國國民黨"},
            {"層級": "高雄市議員", "選區": "第4選區（左營、楠梓）", "姓名": "李雅慧", "黨籍": "民主進步黨"},
            {"層級": "高雄市議員", "選區": "第4選區（左營、楠梓）", "姓名": "黃文志", "黨籍": "民主進步黨"},
            {"層級": "高雄市議員", "選區": "第4選區（左營、楠梓）", "姓名": "李眉蓁", "黨籍": "中國國民黨"},
            {"層級": "高雄市議員", "選區": "第4選區（左營、楠梓）", "姓名": "陳玫娟", "黨籍": "中國國民黨"},
            {"層級": "高雄市議員", "選區": "第4選區（左營、楠梓）", "姓名": "陳善慧", "黨籍": "無黨籍"},
            {"層級": "高雄市議員", "選區": "第4選區（左營、楠梓）", "姓名": "白喬茵", "黨籍": "中國國民黨"},
            {"層級": "高雄市議員", "選區": "第4選區（左營、楠梓）", "姓名": "陳麗珍", "黨籍": "中國國民黨"},
            {"層級": "高雄市議員", "選區": "第4選區（左營、楠梓）", "姓名": "湯詠瑜", "黨籍": "民主進步黨"},
            {"層級": "高雄市議員", "選區": "第5選區（仁武、大社等）", "姓名": "黃飛鳳", "黨籍": "民主進步黨"},
            {"層級": "高雄市議員", "選區": "第8選區（前金、新興、苓雅）", "姓名": "郭建盟", "黨籍": "民主進步黨"},
            {"層級": "高雄市議員", "選區": "第7選區（三民）", "姓名": "張博洋", "黨籍": "台灣基進"},
        ]
    return councilors


def get_kaohsiung_village_chiefs():
    """橋頭區 17 里里長與相關聯誼會代表（可自行擴充全高雄市 890 里 CSV）"""
    # 橋頭區 17 里：橋頭、橋南、仕隆、仕豐、仕和、筆秀、東林、西林、甲北、甲南、白樹、芋寮、頂鹽、新莊、德松、中崎、三德
    # 若要擴大至全高雄 890 位里長，可於下方載入政府開放資料平台「村里長名冊」CSV
    qiaotou_villages = [
        "橋頭里", "橋南里", "仕隆里", "仕豐里", "仕和里", "筆秀里",
        "東林里", "西林里", "甲北里", "甲南里", "白樹里", "芋寮里",
        "頂鹽里", "新莊里", "德松里", "中崎里", "三德里"
    ]
    chiefs = [
        {"層級": "里長", "選區": "橋頭區（里長聯誼會／整體）", "姓名": "橋頭區里長聯誼會", "黨籍": "跨黨派/無黨"},
        {"層級": "里長", "選區": "橋頭區橋南里", "姓名": "蘇道明", "黨籍": "無黨籍"},
        {"層級": "里長", "選區": "橋頭區中崎里", "姓名": "蘇招治", "黨籍": "無黨籍"},
    ]
    for v in qiaotou_villages:
        chiefs.append({
            "層級": "里長（以里名檢索）",
            "選區": f"橋頭區{v}",
            "姓名": f"橋頭 {v} 里長",
            "黨籍": "地方基層",
        })
    return chiefs


def get_2026_councilor_candidates():
    """115年（2026年）高雄市議員候選人（含第3選區全體8人與第4選區全體12人）"""
    return [
        # 第3選區（永安、岡山、燕巢、彌陀、梓官、橋頭，應選5席，8人參選）
        {"層級": "115年市議員候選人（現任）", "選區": "第3選區（橋頭、岡山等）1號", "姓名": "方信淵", "黨籍": "中國國民黨"},
        {"層級": "115年市議員候選人（新人）", "選區": "第3選區（橋頭、岡山等）2號", "姓名": "張騏晟", "黨籍": "無黨籍"},
        {"層級": "115年市議員候選人（現任）", "選區": "第3選區（橋頭、岡山等）3號", "姓名": "宋立彬", "黨籍": "中國國民黨"},
        {"層級": "115年市議員候選人（新人）", "選區": "第3選區（橋頭、岡山等）4號", "姓名": "蔣黃女", "黨籍": "無黨籍"},
        {"層級": "115年市議員候選人（現任）", "選區": "第3選區（橋頭、岡山等）5號", "姓名": "黃秋媖", "黨籍": "民主進步黨"},
        {"層級": "115年市議員候選人（現任）", "選區": "第3選區（橋頭、岡山等）6號", "姓名": "林志誠", "黨籍": "民主進步黨"},
        {"層級": "115年市議員候選人（新人）", "選區": "第3選區（橋頭、岡山等）7號", "姓名": "黃韻涵", "黨籍": "中國國民黨"},
        {"層級": "115年市議員候選人（新人）", "選區": "第3選區（橋頭、岡山等）8號", "姓名": "蔡秉璁", "黨籍": "民主進步黨"},
        # 第4選區（左營、楠梓，12人參選）
        {"層級": "115年市議員候選人（現任）", "選區": "第4選區（左營、楠梓）", "姓名": "陳麗珍", "黨籍": "中國國民黨"},
        {"層級": "115年市議員候選人（現任）", "選區": "第4選區（左營、楠梓）", "姓名": "李雅芬", "黨籍": "中國國民黨"},
        {"層級": "115年市議員候選人（現任）", "選區": "第4選區（左營、楠梓）", "姓名": "李眉蓁", "黨籍": "中國國民黨"},
        {"層級": "115年市議員候選人（現任）", "選區": "第4選區（左營、楠梓）", "姓名": "白喬茵", "黨籍": "中國國民黨"},
        {"層級": "115年市議員候選人（現任）", "選區": "第4選區（左營、楠梓）", "姓名": "陳玫娟", "黨籍": "中國國民黨"},
        {"層級": "115年市議員候選人（現任）", "選區": "第4選區（左營、楠梓）", "姓名": "黃文志", "黨籍": "民主進步黨"},
        {"層級": "115年市議員候選人（現任）", "選區": "第4選區（左營、楠梓）", "姓名": "李雅慧", "黨籍": "民主進步黨"},
        {"層級": "115年市議員候選人（新人）", "選區": "第4選區（左營、楠梓）", "姓名": "張以理", "黨籍": "民主進步黨"},
        {"層級": "115年市議員候選人（新人）", "選區": "第4選區（左營、楠梓）", "姓名": "尹立", "黨籍": "民主進步黨"},
        {"層級": "115年市議員候選人（新人）", "選區": "第4選區（左營、楠梓）", "姓名": "薛兆基", "黨籍": "無黨籍"},
        {"層級": "115年市議員候選人（新人）", "選區": "第4選區（左營、楠梓）", "姓名": "柳淑芳", "黨籍": "無黨籍"},
        {"層級": "115年市議員候選人（新人）", "選區": "第4選區（左營、楠梓）", "姓名": "陳冠宇", "黨籍": "無黨籍"},
    ]


raw_reps = (
    get_kaohsiung_legislators()
    + get_2026_councilor_candidates()
    + crawl_kaohsiung_councilors()
    + get_kaohsiung_village_chiefs()
)
# 依姓名去重（優先保留候選人詳細標註）
seen_names = set()
reps = []
for r in raw_reps:
    if r["姓名"] not in seen_names:
        seen_names.add(r["姓名"])
        reps.append(r)
df_reps = pd.DataFrame(reps)
print(f"共載入 {len(df_reps)} 筆高雄各級民代／檢索目標：")
display(df_reps["層級"].value_counts())
df_reps.head(15)

### 步驟 4：針對每位民代進行新聞與發言爬蟲
透過 Google News RSS 搜尋「`民代姓名` + 橋頭糖廠／高雄新市鎮三期／1-1道路／森林阿嬤」，並抓取前幾則新聞的正文進行關鍵字段落擷取與立場初判。
> **說明**：若該民代搜尋結果為 0 筆，將標記為「未見公開表態／無新聞紀錄」。

In [ ]:
# @title 4-1 執行爬蟲：搜尋新聞、擷取發言片段與立場初判

def search_google_news(person_name, max_results=5):
    query = f'"{person_name}" {ISSUE_QUERY}'
    rss_url = f"https://news.google.com/rss/search?q={quote(query)}&hl=zh-TW&gl=TW&ceid=TW:zh-Hant"
    feed = feedparser.parse(rss_url)
    items = []
    for entry in feed.entries[:max_results]:
        items.append({
            "title": entry.get("title", ""),
            "link": entry.get("link", ""),
            "published": entry.get("published", ""),
            "source": entry.get("source", {}).get("title", "") if isinstance(entry.get("source"), dict) else "",
        })
    return items


def fetch_article_snippet(url, person_name, max_chars=280):
    """嘗試解析新聞正文，並擷取包含人名或關鍵字的句子"""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10, allow_redirects=True)
        text = trafilatura.extract(resp.text) or ""
        if not text:
            return ""
        sentences = re.split(r"[。！？\n]", text)
        matched = [
            s.strip() for s in sentences
            if (person_name.split()[0] in s or any(k in s for k in ISSUE_KEYWORDS)) and len(s.strip()) > 10
        ]
        snippet = "。".join(matched[:3])
        return snippet[:max_chars]
    except Exception:
        return ""


def classify_stance(text):
    """依關鍵字初步分類立場（僅供輔助判讀，建議人工複核）"""
    if not text:
        return "未見公開表態"
    pro_forest_kw = ["護樹", "保留森林", "反對開發", "替代方案", "繞道", "守護橋頭", "公聽會", "生態破壞", "森林阿嬤"]
    pro_dev_kw = ["力挺開發", "加速推動", "支持開發", "1-1", "交通瓶頸", "塞車", "翻轉", "產業發展", "沒有三期"]
    neutral_kw = ["協商", "平衡", "兼顧", "公民參與", "公開透明", "聽取"]

    f_score = sum(1 for k in pro_forest_kw if k in text)
    d_score = sum(1 for k in pro_dev_kw if k in text)
    n_score = sum(1 for k in neutral_kw if k in text)

    if f_score > d_score and f_score > 0:
        return "偏護樹／保留或要求檢討"
    if d_score > f_score and d_score > 0:
        return "偏支持開發／加速交通建設"
    if n_score > 0 or (f_score > 0 and d_score > 0):
        return "折衷／呼籲溝通或兼顧"
    return "有相關報導（立場待人工判讀）"


results = []
for _, row in tqdm(df_reps.iterrows(), total=len(df_reps), desc="爬取民代發言紀錄"):
    name = row["姓名"]
    news_list = search_google_news(name, max_results=4)
    
    snippets = []
    titles = []
    links = []
    dates = []
    for item in news_list[:2]:  # 針對前 2 則深入抓內文以節省時間
        titles.append(item["title"])
        links.append(item["link"])
        dates.append(item["published"])
        snip = fetch_article_snippet(item["link"], name)
        if snip:
            snippets.append(snip)
        time.sleep(0.4)

    combined_text = " ".join(titles + snippets)
    stance = classify_stance(combined_text) if news_list else "未見公開表態"

    results.append({
        "層級": row["層級"],
        "選區": row["選區"],
        "姓名": name,
        "黨籍": row["黨籍"],
        "報導數": len(news_list),
        "初步立場判讀": stance,
        "代表性報導標題": " | ".join(titles),
        "關鍵發言/內文摘要": " // ".join(snippets),
        "來源連結": " ; ".join(links),
    })
    time.sleep(0.5)

df_result = pd.DataFrame(results)
df_result.head(20)

### 步驟 5：（選用）使用 Gemini API 進行精準語意立場分析
如果您有 Gemini API Key，可填入下方，讓模型針對抓到的標題與發言摘要做更精準的「立場分類與一句話總結」。若不填寫可跳過此步，直接執行步驟 6 匯出。

In [ ]:
# @title 5-1（選用）Gemini 語意立場總結
GEMINI_API_KEY = ""  # @param {type:"string"}

if GEMINI_API_KEY.strip():
    !pip install -q google-genai
    from google import genai
    client = genai.Client(api_key=GEMINI_API_KEY.strip())

    ai_summaries = []
    for _, r in tqdm(df_result.iterrows(), total=len(df_result), desc="AI 判讀中"):
        if r["報導數"] == 0:
            ai_summaries.append("無公開報導紀錄")
            continue
        prompt = (
            f"請根據以下關於高雄民代「{r['姓名']}」（{r['層級']}，{r['選區']}）在「橋頭糖廠森林／高雄新市鎮三期開發」議題的報導與摘要，"
            "用繁體中文回答：1) 立場分類（反對開發護樹 / 支持開發交通 / 折衷呼籲協商 / 僅轉述未表態）；2) 50字內核心發言重點。\n"
            f"標題：{r['代表性報導標題']}\n摘要：{r['關鍵發言/內文摘要']}"
        )
        try:
            resp = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
            ai_summaries.append(resp.text.strip())
        except Exception as e:
            ai_summaries.append(f"AI 呼叫失敗: {e}")
        time.sleep(0.5)
    df_result["AI立場與發言總結"] = ai_summaries
else:
    print("未填入 GEMINI_API_KEY，略過 AI 語意總結，保留關鍵字初步判讀結果。")

In [ ]:
# @title 步驟 6：匯出 CSV / JSON 並繪製表態分布統計
import matplotlib.pyplot as plt

csv_path = "kaohsiung_reps_qiaotou_forest.csv"
json_path = "kaohsiung_reps_qiaotou_forest.json"

df_result.to_csv(csv_path, index=False, encoding="utf-8-sig")
df_result.to_json(json_path, orient="records", force_ascii=False, indent=2)

print(f"已匯出：{csv_path} 與 {json_path}")
print("\n各立場統計：")
print(df_result["初步立場判讀"].value_counts())

# 在 Colab 中下載檔案
try:
    from google.colab import files
    files.download(csv_path)
    files.download(json_path)
except ImportError:
    print("非 Colab 環境，檔案已儲存於當前工作目錄。")